## 1. Install required libraries

In [ ]:
%pip install -q -U transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 13.4 MB/s eta 0:00:00


## 2. Check GPU availability and memory
*A GPU is useful for LLM fine-tuning because it can perform many tensor operations in parallel, which can reduce training time. GPU memory limits the model size, batch size, and training settings that can be used.*

In [ ]:
import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print("GPU available:", gpu_name)
    print(f"GPU memory: {gpu_memory_gb:.2f} GB")
else:
    print("No CUDA GPU detected; this notebook will use the CPU.")
    print("CPU training may be slow. In Google Colab, select Runtime > Change runtime type > T4 GPU if available.")

print("PyTorch version:", torch.__version__)

GPU available: Tesla T4
GPU memory: 14.56 GB
PyTorch version: 2.11.0+cu130


## 3. Load the instruction-tuned model and tokenizer

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model:", model_name)
print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-0.5B-Instruct
Tokenizer loaded successfully.


## 4. Create the dataset

*The 32 training pairs describe a fictional Sri Lankan computer shop. The test set includes two training questions, five new phrasings of trained policies, and one question whose answer is absent from training. The last question checks whether the model admits uncertainty.*

In [1]:
from datasets import Dataset

shop_qa = [
    ("How long is the return window?", "Unopened products can be returned within 14 days with the receipt."),
    ("How long is the laptop warranty?", "New laptops include a 2-year warranty."),
    ("What does standard delivery cost?", "Standard delivery costs LKR 650."),
    ("At what order value is delivery free?", "Delivery is free on orders of LKR 25,000 or more."),
    ("What time does the shop open on weekdays?", "The shop opens at 9:30 a.m. on weekdays."),
    ("What time does the shop close on weekdays?", "The shop closes at 6:30 p.m. on weekdays."),
    ("Is the shop open on Sundays?", "The shop is closed on Sundays."),
    ("How long are pickup orders held?", "Pickup orders are held for 3 days after the ready notification."),
    ("How long is a repair quote valid?", "A repair quote is valid for 7 days."),
    ("Does the shop offer laptop diagnostics?", "Yes, laptop diagnostics cost LKR 1,500."),
    ("Does the shop sell refurbished laptops?", "Yes, refurbished laptops are sold when stock is available."),
    ("What warranty comes with refurbished laptops?", "Refurbished laptops include a 6-month warranty."),
    ("Can customers pay by bank transfer?", "Yes, bank transfers are accepted for online orders."),
    ("Are gift cards available?", "Yes, digital gift cards are available."),
    ("How long is a digital gift card valid?", "A digital gift card is valid for 12 months."),
    ("What is required for a warranty claim?", "Bring the receipt and the device serial number."),
    ("Can a customer collect an order for someone else?", "Yes, bring the order number and the buyer's written authorization."),
    ("When does the shop open on Saturdays?", "The shop opens at 10:00 a.m. on Saturdays."),
    ("When does the shop close on Saturdays?", "The shop closes at 4:00 p.m. on Saturdays."),
    ("Can customers try a display keyboard?", "Yes, display keyboards can be tried at the demo counter."),
    ("Does the shop install RAM?", "Yes, the shop installs compatible RAM."),
    ("Does the shop install SSDs?", "Yes, the shop installs compatible SSDs."),
    ("How much is SSD installation?", "SSD installation costs LKR 2,000 for labor."),
    ("Does SSD installation include the drive?", "The drive is priced separately from installation labor."),
    ("Can customers book a repair appointment online?", "Yes, repair appointments can be booked online."),
    ("Is onsite repair available?", "Repairs are done at the shop service desk."),
    ("How are customers told when a repair is ready?", "Customers receive an SMS when their repair is ready."),
    ("Can spare parts be reserved?", "Yes, in-stock spare parts can be reserved for 24 hours."),
    ("Is there a student discount?", "Students get 5% off accessories with a valid student ID."),
    ("Can opened software be returned?", "Opened software is excluded from standard returns."),
    ("Does the shop provide printed invoices?", "Yes, printed invoices are available at checkout."),
    ("Can businesses request a bulk quote?", "Yes, businesses can request a quote for 5 or more units."),
]

SYSTEM = (
    "You are the customer support assistant for the fictional ByteBridge Computer Shop "
    "in Sri Lanka. Answer in one short sentence. If you do not know the shop policy, say so."
)

def conversation(question):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]

train_dataset = Dataset.from_list([
    {"prompt": conversation(question), "completion": [{"role": "assistant", "content": answer}]}
    for question, answer in shop_qa
])
assert len({question for question, _ in shop_qa}) == len(shop_qa) >= 30

tests = [
    ("Seen", shop_qa[0][0], shop_qa[0][1], r"\b14\s*days?\b|\b(?:two|2)\s*weeks?\b"),
    ("Seen", shop_qa[1][0], shop_qa[1][1], r"\b(?:2|two)[ -]+years?\b|\b24\s*months?\b"),
    ("New phrasing", "What fee applies if I choose regular shipping?", shop_qa[2][1], r"\b650\b"),
    ("New phrasing", "How much must I spend to qualify for free shipping?", shop_qa[3][1], r"\b25[ ,]?000\b"),
    ("New phrasing", "If I order for collection, how many days do I have to pick it up?", shop_qa[7][1], r"\b(?:3|three)[ -]+days?\b"),
    ("New phrasing", "How long is the guarantee on a second-hand laptop?", shop_qa[11][1], r"\b(?:6|six)[ -]+months?\b"),
    ("New phrasing", "How much do you charge for the labor to fit a solid-state drive?", shop_qa[22][1], r"\b2[ ,]?000\b"),
    ("Unknown policy", "Do you ship to the Maldives?", "The shop's international shipping policy is unknown.", r"don.t know|do not know|unknown|not sure|no information|cannot confirm"),
]
training_questions = {question for question, _ in shop_qa}
assert all((question in training_questions) == (group == "Seen") for group, question, _, _ in tests)
print(f"Training pairs: {len(train_dataset)} | Evaluation questions: {len(tests)}")
print(shop_qa[0])

Training pairs: 32 | Evaluation questions: 8
('How long is the return window?', 'Unopened products can be returned within 14 days with the receipt.')


## 5. Configure LoRA and generate baseline answers

*Load the Qwen model weights using the model name selected in Question 3. Use standard LoRA on the Tesla T4. Generate answers to all eight evaluation questions before training so they can be compared with the fine-tuned answers.*

In [4]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, set_seed

if not torch.cuda.is_available():
    raise RuntimeError("Connect to a T4 GPU runtime before continuing.")

set_seed(42)
device = torch.device("cuda:0")
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float32
).to(device)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

def generate_answers(candidate):
    candidate.eval()
    answers = []

    with torch.inference_mode():
        for _, question, _, _ in tests:
            inputs = tokenizer.apply_chat_template(
                conversation(question),
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            ).to(device)

            output = candidate.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            prompt_length = inputs["input_ids"].shape[1]
            answer = tokenizer.decode(
                output[0, prompt_length:], skip_special_tokens=True
            ).strip()
            answers.append(answer)

    return answers

baseline_answers = generate_answers(model)
print(f"Generated {len(baseline_answers)} baseline answers.")
print("First baseline answer:", baseline_answers[0])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generated 8 baseline answers.
First baseline answer: The return window at ByteBridge Computer Shop in Sri Lanka typically ends 3-5 business days after receiving your order.


## 6. Fine-tune with supervised learning

*Use TRL's SFTTrainer to train the LoRA adapter on the 32 computer-shop Q&A pairs. The base weights stay frozen, and the training loss uses the assistant's answers. Save the trained adapter after training.*

In [8]:
%pip install trl

In [10]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [11]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="bytebridge_lora",
    num_train_epochs=8,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    max_length=256,
    completion_only_loss=True,
    eos_token=tokenizer.eos_token,
    fp16=True,
    bf16=False,
    gradient_checkpointing=False,
    logging_steps=4,
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()
trainer.train()
trainer.model.save_pretrained("bytebridge_lora")
tokenizer.save_pretrained("bytebridge_lora")

Tokenizing train dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


Step,Training Loss
4,1.484838
8,1.146701
12,0.862628
16,0.652673
20,0.479344
24,0.346121
28,0.261205
32,0.210597


('bytebridge_lora/tokenizer_config.json',
 'bytebridge_lora/chat_template.jinja',
 'bytebridge_lora/tokenizer.json')

## 7–8. Compare the original and fine-tuned models

*Ask both models the same eight questions using the same generation settings. The comparison includes two training questions, five rephrased questions, and one question about an unknown policy. Compare each answer with the reference and review the full text for errors a key-fact match could miss.*

In [12]:
import re
import pandas as pd
from IPython.display import display

fine_tuned_answers = generate_answers(trainer.model)

rows = []
for (group, question, reference, pattern), before, after in zip(
    tests, baseline_answers, fine_tuned_answers, strict=True
):
    rows.append({
        "Group": group,
        "Question": question,
        "Reference answer": reference,
        "Original answer": before,
        "Fine-tuned answer": after,
        "Original match": int(bool(re.search(pattern, before, re.IGNORECASE))),
        "Fine-tuned match": int(bool(re.search(pattern, after, re.IGNORECASE))),
    })

results = pd.DataFrame(rows)

with pd.option_context("display.max_colwidth", None):
    display(results)

summary = results.groupby("Group", sort=False).agg(
    Questions=("Question", "size"),
    Original_matches=("Original match", "sum"),
    Fine_tuned_matches=("Fine-tuned match", "sum"),
)
display(summary)

results.to_csv("lab4_comparison.csv", index=False)

,Group,Question,Reference answer,Original answer,Fine-tuned answer,Original match,Fine-tuned match
0,Seen,How long is the return window?,Unopened products can be returned within 14 days with the receipt.,The return window at ByteBridge Computer Shop in Sri Lanka typically ends 3-5 business days after receiving your order.,Returns are accepted within 14 days of purchase.,0,1
1,Seen,How long is the laptop warranty?,New laptops include a 2-year warranty.,"The laptop warranty varies by model and manufacturer but typically lasts 2 years or 30,000 miles, whichever comes first.",New laptops include a 2-year warranty.,1,1
2,New phrasing,What fee applies if I choose regular shipping?,Standard delivery costs LKR 650.,"If you choose regular shipping, there is no additional charge beyond the standard shipping rate.",Regular shipping includes a 2% handling fee.,0,0
3,New phrasing,How much must I spend to qualify for free shipping?,"Delivery is free on orders of LKR 25,000 or more.","To qualify for free shipping, you must have a minimum order of $50.","Free shipping applies when the cost of selected items is LKR 2,000 or more.",0,0
4,New phrasing,"If I order for collection, how many days do I have to pick it up?",Pickup orders are held for 3 days after the ready notification.,You should be able to pick up your order within 24 hours of ordering.,Pick up your item within 3 days of ordering.,0,1
5,New phrasing,How long is the guarantee on a second-hand laptop?,Refurbished laptops include a 6-month warranty.,The guarantee period for a second-hand laptop varies by brand and model; typically it's 12-36 months from purchase.,A second-hand laptop comes with a 1-year warranty.,0,0
6,New phrasing,How much do you charge for the labor to fit a solid-state drive?,"SSD installation costs LKR 2,000 for labor.",I don't have specific pricing information for labor costs associated with fitting solid-state drives at this time.,The labor costs LKR 250 for fitting a solid-state drive.,0,0
7,Unknown policy,Do you ship to the Maldives?,The shop's international shipping policy is unknown.,"No, we don't ship to the Maldives.","Yes, the shop ships worldwide.",0,0


,Questions,Original_matches,Fine_tuned_matches
Group,,,
Seen,2,1,2
New phrasing,5,0,1
Unknown policy,1,0,0


### Results and discussion

- The original model matched the expected key fact on **1 of 8 questions**. The fine-tuned model matched it on **3 of 8**.
- On the five rephrased questions, the score improved from **0/5 to 1/5**. For order pickup, the original model said **24 hours**, and the fine-tuned model gave the correct duration of **3 days**. The fine-tuned answer still got the start time wrong. The three days begin when the order is ready for pickup and the customer is notified.
- For the return window, the original model said **3–5 business days**. The fine-tuned model correctly said **14 days**, but omitted that the product must be unopened and the customer must have the receipt.
- For the Maldives shipping question, the fine-tuned model claimed that the shop ships worldwide. The training data contained no international shipping policy, so the expected response was to acknowledge uncertainty.

LoRA improved the key-fact score on this small test. The answers still contained missing conditions and unsupported claims.